# Introduction
We're going to analyze a dataset about the westbound traffic on the I-94 Interstate Highway. The dataset is available [here](https://archive.ics.uci.edu/ml/datasets/Metro+Interstate+Traffic+Volume).
The goal of our analysis is to determine a few indicators of heavy traffic on I-94. These indicators can be weather type, time of the day, time of the week, etc. For instance, we may find out that the traffic is usually heavier depending on the season or the weather.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('Metro_Interstate_Traffic_Volume.csv')
print(df.head())
print(df.tail())
print(df.info())

We are goign to observe the distribution of the `traffic_volume`column.

In [ ]:
df['traffic_volume'].plot.hist()
plt.show()

In [ ]:
df['traffic_volume'].describe()

Every hour, the traffic volume varied from 0 to 7280 cars, with an average of 3260 cars. About 25% of the time, there were only 1193 cars or fewer passing the station each hour (this probably occurs during the night, or when a road is under construction). However, about 25% of the time, the traffic volume was four times as much (4933 cars or more).

This observation gives our analysis an interesting direction: comparing daytime data with nighttime data.

# Trafic volume: night vs day
We'll start by dividing the dataset into two parts:

 - Daytime data: hours from 7 AM to 7 PM (12 hours)
 - Nighttime data: hours from 7 PM to 7 AM (12 hours)
 
For this we will use the `Series.dt.hour` property.

In [ ]:
df['date_time'] = pd.to_datetime(df['date_time'])  # transform data into datetime objects

In [ ]:
day = df[(df['date_time'].dt.hour >= 7) & (df['date_time'].dt.hour < 19)]
night = df[(df['date_time'].dt.hour >= 19) | (df['date_time'].dt.hour < 7)]

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(day['traffic_volume'])
plt.title('traffic volume: day')
plt.ylabel('frequency')
plt.xlabel('traffic volume')
plt.xlim(-100, 7500)
plt.ylim(0, 8000)

plt.subplot(1,2,2)
plt.hist(night['traffic_volume'])
plt.title('traffic volume: night')
plt.ylabel('frequency')
plt.xlabel('traffic volume')
plt.xlim(-100, 7500)
plt.ylim(0, 8000)

plt.show()

In [ ]:
day['traffic_volume'].describe()

In [ ]:
night['traffic_volume'].describe()

The histogram that shows the distribution of traffic volume during the day is left skewed. This means that most of the traffic volume values are high: there are 4252 or more cars passing the station each hour 75% of the time (because 25% of values are less than 4252).

The histogram displaying the nighttime data is right skewed. This means that most of the traffic volume values are low: 75% of the time, the number of cars that passed the station each hour was less than 2,819.

Although there are still measurements of over 5000 cars per hour, the traffic at night is generally light. Our goal is to find indicators of heavy traffic, so we'll only focus on the daytime data moving forward.

# Time indicator
One of the possible indicators of heavy traffic is time. There might be more people on the road in a certain month, on a certain day, or at a certain time of the day.

We are going to plot some graphs to understand how the traffic volume changed with different parameters: month, day of the week, time.

In [ ]:
day['month'] = day['date_time'].dt.month
by_month = day.groupby('month').mean()
by_month['traffic_volume']


In [ ]:
by_month['traffic_volume'].plot()
plt.show()


The traffic looks less heavy during cold months (November to February) and more intense during warm months (March to October), with one interesting exception: July. Is there anything special about July? Is traffic significantly less heavy in July each year? One possible reason for this is road construction that occured at this time.

We can conclude that warm months generally show heavier traffic compared to cold months. In a warm month, you can can expect for each hour of daytime a traffic volume close to 5000 cars.

In [ ]:
day['dayofweek'] = day['date_time'].dt.dayofweek
by_dayofweek = day.groupby('dayofweek').mean()
by_dayofweek['traffic_volume'] #0 is Monday, 6 is Sunday

In [ ]:
by_dayofweek['traffic_volume'].plot()
plt.show()

Traffic volume is significantly heavier on business days (Monday to Friday). Except for Monday, we only see values over 5000 during business days. Traffic is lighter on weekends, with values below 4000 cars

We want now to study the traffic by hour, but the weekends are goign to modify the avergage, so we need to separate business days vs weekend.

In [ ]:
day['hour'] = day['date_time'].dt.hour
business_days = day.copy()[day['dayofweek'] <= 4] # 4 is Friday
weekend = day.copy()[day['dayofweek'] >= 5] # 5 is Saturday
by_hour_business = business_days.groupby('hour').mean()
by_hour_weekend = weekend.groupby('hour').mean()

print(by_hour_business['traffic_volume'])
print(by_hour_weekend['traffic_volume'])

In [ ]:
plt.figure(figsize=(11,3.5))
plt.subplot(1,2,1)
by_hour_business['traffic_volume'].plot()
plt.xlim(6,20)
plt.ylim(1500,6500)
plt.title('traffic by hour for business days')

plt.subplot(1,2,2)
by_hour_weekend['traffic_volume'].plot()
plt.xlim(6,20)
plt.ylim(1500,6500)
plt.title('traffic by hour for weekend')

plt.show()

At each hour of the day, the traffic volume is generally higher during business days compared to the weekends. The rush hours are around 7 and 16 — when most people travel from home to work and back. We see volumes of over 6000 cars at rush hours.

To summarize, we found a few time-related indicators of heavy traffic:

 - The traffic is usually heavier during warm months (March–October) compared to cold months (November–February).
 - The traffic is usually heavier on business days compared to weekends.
 - On business days, the rush hours are around 7 and 16.

# Weather indicator

Another possible indicator of heavy traffic is weather. The dataset provides us with a few useful columns about weather: `temp`, `rain_1h`, `snow_1h`, `clouds_all`, `weather_main`, `weather_description`.

A few of these columns are numerical so let's start by looking up their correlation values with `traffic_volume`.

In [ ]:
day.corr()['traffic_volume']

Temperature shows the strongest correlation with a value of just +0.13. The other relevant columns (`rain_1h`, `snow_1h`, `clouds_all`) don't show any strong correlation with `traffic_value`.

Let's observe visually the correlation between `temp` and `traffic_volume`

In [ ]:
day.plot.scatter('traffic_volume', 'temp')
plt.ylim(230,330) # needed because two wrong temperatures mess the y axis
plt.show()

We can conclude from this graph that there is no correlation and that the temperatue is not a relevant indicator of heavy traffic.

# Weather types
Let's now look at the other weather-related columns: `weather_main` and `weather_description`.

To start, we're going to group the data by `weather_main` and look at the `traffic_volume` averages.

In [ ]:
by_weather_main = day.groupby('weather_main').mean()
by_weather_main['traffic_volume'].plot.barh()
plt.show()

It looks like there's no weather type where traffic volume exceeds 5,000 cars. This makes finding a heavy traffic indicator more difficult. Let's also group by `weather_description`, which has a more granular weather classification.

In [ ]:
by_weather_description = day.groupby('weather_description').mean()
by_weather_description['traffic_volume'].plot.barh(figsize=(6,11))

plt.show()

It looks like some values are over 5000. Let's take a look at it.

In [ ]:
by_weather_description[by_weather_description['traffic_volume'] >= 5000]['traffic_volume'].plot.barh()
plt.show()

It's not clear why these weather types have the highest average traffic values — this is bad weather, but not that bad. Perhaps more people take their cars out of the garage when the weather is bad instead of riding a bike or walking.

# Conclusion
In this project, we tried to find a few indicators of heavy traffic on the I-94 Interstate highway. We managed to find two types of indicators:

Time indicators:
 - The traffic is usually heavier during warm months (March–October) compared to cold months (November–February).
 - The traffic is usually heavier on business days compared to the weekends.
 - On business days, the rush hours are around 7 and 16.
 
Weather indicators:
 - Shower snow
 - Light rain and snow
 - Proximity thunderstorm with drizzle